# AccentSense — UK Regional Accent Classifier
**WavLM Base+ · Attentive Statistics Pooling · 7 UK Dialect Classes**

VIT Bhopal University · AI/ML Capstone Research

---
**Classes:** `RP` · `Scottish` · `Welsh` · `Northern` · `West_Midlands` · `Cockney` · `Irish`

**Hardware:** Colab T4 GPU (Runtime → Change runtime type → T4)

**Steps:**
1. GPU check
2. Install dependencies
3. Clone repo
4. Download VCTK corpus (real UK speakers, 16 kHz)
5. (Optional) Mozilla Common Voice Irish supplement
6. Build speaker-disjoint manifests
7. Train WavLM (20 epochs, cosine LR, class-weighted CE)
8. Evaluate — classification report + confusion matrix
9. Download checkpoint → place in `Backend/checkpoints/`


In [ ]:
# Step 1: Verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('VRAM: %.1f GB' % vram)
else:
    print('WARNING: No GPU. Go to Runtime -> Change runtime type -> T4 GPU')


In [ ]:
# Step 2: Install dependencies
!pip install -q transformers accelerate torchaudio datasets soundfile jiwer scikit-learn pandas
print('All dependencies installed.')


In [ ]:
# Step 3: Clone AccentSense repository
import os
# UPDATE: replace with your actual GitHub repo
REPO = 'SameerGera/Accent-Sense'
!git clone https://github.com/{REPO}.git AccentSense
os.chdir('AccentSense/Backend')
import sys
sys.path.insert(0, '.')
print('Working dir:', os.getcwd())


In [ ]:
# Step 4: Download VCTK Corpus (University of Edinburgh, free)
# Fixed URL: datashare.is.ed.ac.uk (not datashare.ed.ac.uk)
# Falls back to Hugging Face if direct download fails
import os, glob

VCTK_URL = 'https://datashare.is.ed.ac.uk/bitstream/handle/10283/3443/VCTK-Corpus-0.92.zip'
USE_HF_VCTK = False
zip_path = 'data/vctk.zip'
os.makedirs('data/vctk', exist_ok=True)

# Attempt direct download with retries
try:
    !wget -q --tries=3 --timeout=60 {VCTK_URL} -O {zip_path}
    zip_size = os.path.getsize(zip_path) / (1024**3)
    print(f'Downloaded: {zip_size:.2f} GB')
    if zip_size < 10.0:
        print('WARNING: File too small, download may be corrupt. Retrying...')
        os.remove(zip_path)
        !wget -q --tries=3 {VCTK_URL} -O {zip_path}
        zip_size = os.path.getsize(zip_path) / (1024**3)
except Exception as e:
    print(f'Direct download failed: {e}')

# Validate and extract
if os.path.exists(zip_path) and os.path.getsize(zip_path) / (1024**3) > 10.0:
    print('Testing zip integrity...')
    !unzip -t {zip_path}
    print('Extracting VCTK...')
    !cd data && unzip -q vctk.zip -d vctk/ && rm vctk.zip
    info = glob.glob('data/vctk/**/speaker-info.txt', recursive=True)
    print('speaker-info.txt:', info)
    if not info:
        print('WARNING: speaker-info.txt not found after extraction!')
        USE_HF_VCTK = True
else:
    print('Direct download failed or file corrupt. Falling back to Hugging Face...')
    USE_HF_VCTK = True

if USE_HF_VCTK:
    print('VCTK will be loaded from Hugging Face in Step 6.')


In [ ]:
# Step 5 (Optional): Add Irish English from Mozilla Common Voice
# This supplements VCTK which has very few Irish speakers.
from datasets import load_dataset
import soundfile as sf
import numpy as np

cv = load_dataset(
    'mozilla-foundation/common_voice_17_0', 'en',
    split='train', streaming=True, trust_remote_code=True
)
IRISH_TAGS = {'Ireland', 'Irish', 'Dublin', 'Northern Ireland'}
irish = []
for s in cv.take(100000):
    if s.get('accent', '') in IRISH_TAGS:
        irish.append(s)
    if len(irish) >= 500:
        break
print('Irish clips found:', len(irish))

os.makedirs('data/cv_irish', exist_ok=True)
cv_extras = []
for i, s in enumerate(irish):
    p = 'data/cv_irish/irish_%04d.wav' % i
    sf.write(p, np.array(s['audio']['array'], dtype=np.float32), s['audio']['sampling_rate'])
    cv_extras.append({'path': p, 'label': 'Irish', 'speaker': s.get('client_id', 'cv%d' % i), 'class_idx': 6})
print('Saved %d Irish WAV files.' % len(cv_extras))


In [ ]:
# Step 6: Build speaker-disjoint manifests
from src.data.vctk_dataset import (
    build_vctk_manifest, build_vctk_manifest_from_hf,
    speaker_disjoint_splits, save_manifest, load_manifest,
    CLASSES, CLASS_TO_IDX
)
from collections import Counter

if USE_HF_VCTK:
    from datasets import load_dataset as hf_load_dataset
    print('Loading VCTK from Hugging Face...')
    vctk_hf = hf_load_dataset('CSTR-Edinburgh/vctk', split='train', trust_remote_code=True)
    manifest = build_vctk_manifest_from_hf(vctk_hf)
else:
    INFO = glob.glob('data/vctk/**/speaker-info.txt', recursive=True)[0]
    WAVD = (glob.glob('data/vctk/**/wav16', recursive=True)
            or glob.glob('data/vctk/**/wav48', recursive=True))[0]
    print('Speaker info:', INFO)
    print('WAV dir:', WAVD)
    manifest = build_vctk_manifest(wav_dir=WAVD, speaker_info_path=INFO)

manifest.extend(cv_extras)   # add Common Voice Irish
print('Total samples:', len(manifest))

dist = Counter(e['label'] for e in manifest)
for cls in CLASSES:
    print('  %-15s: %4d' % (cls, dist.get(cls, 0)))

train_m, val_m, test_m = speaker_disjoint_splits(manifest)
print('Splits -> Train: %d | Val: %d | Test: %d' % (len(train_m), len(val_m), len(test_m)))

os.makedirs('data/manifests', exist_ok=True)
save_manifest(train_m, 'data/manifests/train.json')
save_manifest(val_m,   'data/manifests/val.json')
save_manifest(test_m,  'data/manifests/test.json')


In [ ]:
# Step 7: Training (20 epochs, cosine LR, class-weighted CrossEntropyLoss)
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, balanced_accuracy_score
from src.data.vctk_dataset import UKAccentDataset, collate_pad
from src.models.wavlm_classifier import WavLMAccentClassifier

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
N       = len(CLASSES)   # 7
EPOCHS  = 20
BATCH   = 8
LR      = 3e-4

model = WavLMAccentClassifier(
    pretrained_model_name='microsoft/wavlm-base-plus',
    num_classes=N,
    freeze_encoder=True,
    unfreeze_top_k_layers=2,
    dropout_p=0.3,
).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable: %d / %d (%.1f%%)' % (trainable, total, 100*trainable/total))

tr_ds = UKAccentDataset(load_manifest('data/manifests/train.json'), augment=True)
va_ds = UKAccentDataset(load_manifest('data/manifests/val.json'),   augment=False)
tr_dl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,  collate_fn=collate_pad, num_workers=2, pin_memory=True)
va_dl = DataLoader(va_ds, batch_size=BATCH, shuffle=False, collate_fn=collate_pad, num_workers=2)

# Class-weighted loss to handle fewer Cockney/West_Midlands speakers
cnt = Counter(e['class_idx'] for e in load_manifest('data/manifests/train.json'))
w   = torch.tensor([1.0 / cnt.get(i, 1) for i in range(N)], dtype=torch.float32)
w   = w / w.sum() * N
criterion = torch.nn.CrossEntropyLoss(weight=w.to(DEVICE))

optimizer  = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
T          = len(tr_dl) * EPOCHS
scheduler  = get_cosine_schedule_with_warmup(optimizer, T // 10, T)

best = 0.0
os.makedirs('checkpoints', exist_ok=True)

for ep in range(1, EPOCHS + 1):
    model.train()
    el, pp, ll = 0.0, [], []
    for wav, lb in tr_dl:
        wav, lb = wav.to(DEVICE), lb.to(DEVICE)
        optimizer.zero_grad()
        out  = model(wav)
        loss = criterion(out['logits'], lb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        el += loss.item()
        pp.extend(out['logits'].argmax(-1).cpu().tolist())
        ll.extend(lb.cpu().tolist())
    tf = f1_score(ll, pp, average='macro', zero_division=0)

    model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for wav, lb in va_dl:
            out = model(wav.to(DEVICE))
            vp.extend(out['logits'].argmax(-1).cpu().tolist())
            vl.extend(lb.tolist())
    vf  = f1_score(vl, vp, average='macro', zero_division=0)
    vba = balanced_accuracy_score(vl, vp)

    print('Ep %2d/%d Loss %.4f TrF1 %.4f ValF1 %.4f BalAcc %.4f' % (ep, EPOCHS, el/len(tr_dl), tf, vf, vba))
    if vf > best:
        best = vf
        torch.save(model.state_dict(), 'checkpoints/best_wavlm_accentsense.pt')
        print('  --> best %.4f saved' % vf)

print('Training complete. Best Val Macro-F1: %.4f' % best)


In [ ]:
# Step 8: Test-set evaluation
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load('checkpoints/best_wavlm_accentsense.pt', map_location=DEVICE))
model.eval()

te_ds = UKAccentDataset(load_manifest('data/manifests/test.json'), augment=False)
te_dl = DataLoader(te_ds, batch_size=BATCH, shuffle=False, collate_fn=collate_pad)

tp, tl = [], []
with torch.no_grad():
    for wav, lb in te_dl:
        out = model(wav.to(DEVICE))
        tp.extend(out['logits'].argmax(-1).cpu().tolist())
        tl.extend(lb.tolist())

print('=' * 60)
print('TEST SET RESULTS')
print('=' * 60)
print(classification_report(tl, tp, target_names=CLASSES))
print('Confusion Matrix:')
print(confusion_matrix(tl, tp))
test_f1 = f1_score(tl, tp, average='macro', zero_division=0)
print('Final Test Macro-F1: %.4f' % test_f1)


In [ ]:
# Step 9: Download checkpoint
from google.colab import files
ckpt = 'checkpoints/best_wavlm_accentsense.pt'
size = os.path.getsize(ckpt) / 1024 / 1024
print('Downloading checkpoint (%.1f MB)...' % size)
files.download(ckpt)
print('Place at: Backend/checkpoints/best_wavlm_accentsense.pt')
print('Restart uvicorn -> GET /health shows model_loaded: true')


## After Training

1. Download `best_wavlm_accentsense.pt` from Step 9
2. Copy to `Backend/checkpoints/best_wavlm_accentsense.pt`
3. Restart backend: `uvicorn src.api.main:app --reload`
4. `GET /health` → `model_loaded: true`, `supported_classes: [RP, Scottish, Welsh, Northern, West_Midlands, Cockney, Irish]`

**Expected Test Macro-F1:** >0.70 on VCTK real recordings

> **Cockney tip:** VCTK has few London speakers. If Cockney F1 is low (<0.5),
> add more Common Voice clips tagged 'London' in Step 5 and re-run splits.
